# Real — eólica onshore Alemania (SMARD) · E_01 datos y contrato conjunto

Representación, blanqueo por la Gram, datasets de dos rezagos funcionales e hiperparámetros. Escribe el contrato que lee `matlab/psbp_fd_iteracion_mv.m`.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
RAIZ = Path.cwd().resolve().parent
sys.path.insert(0, str(RAIZ))
from mv_psbp.pipeline import *
from mv_psbp import datos, convergencia, evaluacion, comparacion
pd.set_option("display.width", 200)


In [ ]:
# [CONFIG]
NOMBRE    = "eolica_onshore_DE"
INSUMO    = RAIZ / "data" / "reales" / "insumos" / "eolica_onshore_DE_smard.npz"
EID       = "mv_real_eolica_onshore_DE_v01_r01_K12"
DOMINIO   = "reales"
N_CHAINS  = 3
NB_MAX    = 12      # tope de funciones B-spline: p = K * L entra al gating por distancia
MCMC      = {"nsim": 2500, "burn": 500, "N": 30, "M": 50}   # con n = 3007 las cadenas convergen con 2500 (ver E_03)
LIMPIAR   = False


In [ ]:
o = paso01_real(NOMBRE, INSUMO, nb_max=NB_MAX, limpiar=LIMPIAR, mcmc_config=MCMC,
                meta={"fuente": "SMARD CC BY 4.0", "variable": "factor de planta eolica onshore"})
paths = o["paths"]
print(o["eid"]); print("K =", o["K"], " base =", o["fr"].n_basis, o["fr"].order, " T0 =", o["hp"]["partition"]["T0"], " p =", o["hp"]["p"])
display(o["df_gcv"].nsmallest(5, "gcv_mean"))
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].plot(o["df"]["t"], o["TW"][o["df"]["t"], 0], lw=0.5); ax[0].axvline(o["hp"]["partition"]["T0"], color="r", ls="--"); ax[0].set_title("theta_w[:, 1]")
ax[1].plot(o["grilla"] if "grilla" in o else np.linspace(0, 1, 96), o["fr"].reconstruct(o["THETA"][:40]).T, lw=0.6); ax[1].set_title("40 curvas suavizadas")
plt.show()


In [ ]:
hp = o["hp"]["bloque"] if "bloque" in o["hp"] else o["hp"]   # en la variante bloque, el contrato del bloque conjunto
print(json.dumps({k: hp[k] for k in ("global", "mcmc_config", "n_iter", "seed_base", "q", "p", "partition")}, indent=1))
jobs = registrar_jobs([paths], RAIZ / "artefact" / f"jobs_{EID}.json"); print("jobs ->", jobs)